### Set Up
Importing all the different libraires needed, setting up the paths to each of the files, and the specific coordinates (RA, Dec) of the main object (these are found in the .FITS file title)

In [ ]:
# -- Import Libraries --
import astropy
from astropy.io import fits
from astropy.visualization import ZScaleInterval, ImageNormalize
from astropy.table import Table
from astropy.wcs import WCS

import numpy as np

from scipy.ndimage import shift, zoom

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# -- Path to the different Files --
main_image_cutout = r"FITS\cutout_192.6009_-28.2675.fits"
main_image_psf = r"FITS\copsf_192.6009_-28.2675.fits"
neighbors_image = r"FITS\cat-ls-dr10.fits"

#define the object coordinates we are interested in
main_object_ra = 192.6009
main_object_dec = -28.2675

### Opening and Viewing the FITS Files

#### The Main Cutout
First, we open the main image cutout file and inspect it using the `.info()` function. After that, we go deeper and navigate the header of this file, as well as the data category. From there, we find the r-band, which contains our image under the r-band.

In [ ]:
with fits.open(main_image_cutout) as hdu_list: #use with to safely close the file
    hdu_list.info() 
    header = hdu_list[0].header
    #print(header)
    image_data = hdu_list[0].data #shape: (4, 256, 256)
    r_band = image_data[1] #red is index 1 #this is fluxes

**Plotting** <br>
To plot, we use the ZScaleInterval Image Normalizer, and the gray colormap. The Matplotlib library makes the process extremely easy.

In [ ]:
norm = ImageNormalize(r_band, interval=ZScaleInterval())
plt.figure(figsize=(8,8))
plt.imshow(r_band, origin="lower", norm=norm, cmap='gray')
plt.colorbar()
plt.title('r-band cutout for FITS File coord: RA=192.6009, Dec=-28.2675')
plt.show()

#### The PSF Image
After inspecting the r-band image and its structure, we can look at the downloaded PSF Image, which shows the "shape" of the main object, at coordinates: RA 192.6009, Dec -28.2675.
<br>Again, we open the file and extract the basic information, then dive deeper by looking at the `.data` in index 0.
<br>As the image was not bright enough, we mulitplied this by 644, which was found by dividing the brightest point on the original image by the brightest point on the PSF image. 
<br>The shape (`psf_data.shape`) is the dimentions of the array stored inside the FITS File.<br>
    -> NAXIS1 is 1D Arrays, NAXIS2 is 2D Arrays, NAXIS3 is 3D Arrays.<br>
For plotting, we do the same thing but change the data to the psf's data, and use a different colormap.

In [ ]:
with fits.open(main_image_psf) as psf_hdu_list:
    psf_hdu_list.info()
    psf_data = psf_hdu_list[0].data * 644 #found this multiplication factor by dividing brightest point on og image by the brightest point on psf image

    print('PSF shape:', psf_data.shape) #the dimentions of the array -> NAXIS1 is 1D Arrays, NAXIS2 is 2D Arrays, NAXIS3 is 3D Arrays
    print('PSF sum:', psf_data.sum()) # A PSF that sums to 1.0 means: "if I stamp this at a location and scale it by flux F, the total flux contributed to the image equals F."

plt.figure(figsize=(5,5))
plt.imshow(psf_data, origin='lower', cmap='hot')
plt.colorbar()
plt.title('PSF Model Graph')
plt.show()

#### The Neighbors Image
As we don't have to actually see the image, to open the Neighbors Image, we can use Astropy's `Table` module, by running `Table.read()`. This automatically handles things such as closing the file (no need for a `with` block)<br>
Using Numpy and the Astropy Table, we can look filter and read the different types (such as PSF, EXP, REX, etc.). <br>
From here, we could either sort for PSF sources only, or treat the other identified neighbors sources as PSFs.

In [ ]:
# -- Opening and Inspecting the Neighbors Image --
neighbors = Table.read(neighbors_image) #Table.read already handles the closing of the file so no with block needed

print(np.unique(neighbors['type'])) #check the different exitsting types (e.g., PSF, EXP, REX, etc.)

psf_neighbors_sources = neighbors 
#psf_neighbors_sources = neighbors[neighbors['type'] == b'PSF'] #Filter for PSF types ONLY
print(f"Total Neighbors: {len(neighbors)}")
print(f"Total PSF Sources: {len(psf_neighbors_sources)}")

### World Coordinate System Conversion
The World Coordinate System is what allows for the conversion of pixel coordinates (x, y) to **sky coordinates** (RA, Dec), and vice versa. <br>
In FITS files, a set of keywords is stored <u>in the header</u> describing this transformation.<br>
This conversion can be extremely confusing (because the sky is curved), but the `astropy.wcs` module allows to inser a sky position and outputs which pixels on the image that corresponds to, or vice versa.

In our case, this conversion is needed because the neighbors catalog is in RA and Dec in degrees, but the image for the r_band is a 2D numpy array. <br>
First, we connect to the main image cutout's header. The `.celestial` property automatically removes non-spacial dimentions, dropping the WCS down to a 2D system. <br>
After that, we set the variables `img_ny` and `img_nx` to the size (shape) of our image <br>
<br>
Next, to ensure that everything is aligned correctly, we convert each of the corner pixls to the world coordinate system, achieving a full ky coordinates perimeter covering (from the minimum to maximum Right Ascension and Declination)


In [ ]:
wcs = WCS(header).celestial #connect to main image cutout header to access those keywords
#the .celestial property automatically removes non-spacial dimentions, dropping the WCS down to a 2D system 

img_ny, img_nx = r_band.shape #Set the variables to the size of our image 

#setting corner pixels to wolrd coordinate system
bottom_left = wcs.pixel_to_world(0,0)
bottom_right = wcs.pixel_to_world(img_nx, 0)
top_left = wcs.pixel_to_world(0, img_ny)
top_right = wcs.pixel_to_world(img_nx, img_ny)

ra_min  = min(bottom_left.ra.deg,  bottom_right.ra.deg,  top_left.ra.deg,  top_right.ra.deg)
ra_max  = max(bottom_left.ra.deg,  bottom_right.ra.deg,  top_left.ra.deg,  top_right.ra.deg)
dec_min = min(bottom_left.dec.deg, bottom_right.dec.deg, top_left.dec.deg, top_right.dec.deg)
dec_max = max(bottom_left.dec.deg, bottom_right.dec.deg, top_left.dec.deg, top_right.dec.deg)
#this outputs sky coordinates: RA, Dec (Right Accrension, Declination)

print(f"RA covering: {ra_min:.4f} -> {ra_max:.4f} degrees") 
print(f"DEC covering: {dec_min:.4f} -> {dec_max:.4f} degrees") 

#### Filtering the Neighbor Sources
Once that the area is mapped correctly, we need to filter all the neighboring sources so that we select only those in our specified area. <br>
After the code is executed, xs and ys are arrays that contain all the coordinates of found objects in the specified area.

In [ ]:
# Filtering the neighbor catalog to only sources in the specified RA and Dec coverage
filtered_neighbors_list = (
    (psf_neighbors_sources['ra'] >= ra_min) & (psf_neighbors_sources['ra'] <= ra_max) & (psf_neighbors_sources['dec'] >= dec_min) & (psf_neighbors_sources['dec'] <= dec_max)    
)
filtered_psf_neighbors = psf_neighbors_sources[filtered_neighbors_list]
print(f"PSF sources in this image's field: {len(filtered_psf_neighbors)}")

xs, ys = wcs.world_to_pixel_values(filtered_psf_neighbors['ra'], filtered_psf_neighbors['dec'])
#xs and ys are numpy arrays, one pixel coordinate per source
print(f"xs= {xs}")
print(f"ys = {ys}") #testing to check whether the arrays actually have something stored inside
#these two values are the x and y coordinates for each of the filtered psf neighbors


**Verification of WCS and PSF Sources Identification**<br>
To ensure that everything works, we will plot green circles over each of the found objects by using the `xs` and `ys` arrays.

In [ ]:
# WCS Verification by OVERPLOTTING the different PSF sources on the r-band image
norm = ImageNormalize(r_band, interval=ZScaleInterval())
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(r_band, origin='lower', norm=norm, cmap='bone')
ax.scatter(xs, ys, s=120, facecolors='none', edgecolors='lime', linewidths=1.5)
ax.set_title(f'{len(filtered_psf_neighbors)} PSF neighbors overlaid on r-band image')
plt.show()

### PSF Synthetic Image Creation and Placement

In [ ]:
fluxes = filtered_psf_neighbors['flux_r']
print(fluxes)

#### Excluding the Main Object from the Synthetic Image
As seen from the previous WCS Verification Plot (figure X), the main object is being identified as a neighboring source. If this isn't manually removed from the neighbors table, when creating the synthetic image, the main object will be subtracted too.<br>
<br>
To ensure this doesn't happen, we

In [ ]:
#find and exclude the main object 
main_object_distances = np.sqrt((filtered_psf_neighbors['ra'] - main_object_ra)**2 + (filtered_psf_neighbors['dec'] - main_object_dec)**2) #the distance from every source to the object we're interested in
main_obj_index = np.argmin(main_object_distances) #find the index of the main object by finding the smallest distance

psf_object_KEEP = np.arange(len(filtered_psf_neighbors)) != main_obj_index

xs_without_target = xs[psf_object_KEEP]
ys_without_target = ys[psf_object_KEEP]
fluxes_without_target = filtered_psf_neighbors['flux_r'][psf_object_KEEP]